In [1]:
# ============================================================
# GPU Memory Analysis for ulno_afqmc.py :: prep_afqmc
# ============================================================
# When run_afqmc calls prep_afqmc, the function:
#   1. h1e_uas       -> builds core JK + h1eff on GPU
#   2. DF loop       -> uploads each cderi batch to GPU, transforms to cLAS basis
#   3. compress step -> uploads full cderi_clas (naux x npair) to GPU for SVD
#   4. project step  -> projects compressed Cholesky from cLAS to alpha/beta (CPU)
#   5. del           -> attempts to free, but JAX XLA buffers are NOT freed by Python del
#
# After prep_afqmc returns, the GPU is still nearly full before run_lnoafqmc starts.
# This notebook identifies each cause and shows the fix.
import jax
import jax.numpy as jnp
import numpy as np
import gc

In [ ]:
# ------------------------------------------------------------
# ISSUE 1: Python `del` does NOT free JAX/XLA GPU buffers
# ------------------------------------------------------------
# In prep_afqmc (lines 521-525) there is a cleanup block:
#
#   del cderi_clas, cderi_a, cderi_b
#   del h1e, h1mod_a, h1mod_b
#   del clas_coeff, a2c, b2c
#   del v0_a, v0_b
#
# For numpy arrays this works fine.  For jnp arrays, `del` only
# removes the Python reference. The XLA buffer is reference-counted
# inside JAX's runtime and may not be released until JAX's own GC
# cycle runs — which may never happen before run_lnoafqmc starts.
#
# Demonstration:

def check_gpu_bytes():
    # Works on CUDA backends; prints live bytes allocated in the XLA buffer pool
    backend = jax.lib.xla_bridge.get_backend()
    try:
        return backend.live_executable_size_in_bytes()
    except AttributeError:
        return None

# Allocate a large JAX array, then `del` it
large = jnp.ones((4096, 4096), dtype=jnp.float32)   # ~64 MB
large.block_until_ready()
print(f"After allocation: {jax.devices()[0].memory_stats()['bytes_in_use'] / 1e9:.3f} GB")

# del large 
# gc.collect()
print(f"After del + gc.collect (no clear_caches): {jax.devices()[0].memory_stats()['bytes_in_use'] / 1e9:.3f} GB")

# After clear_caches, XLA is allowed to reuse / release buffers
jax.clear_caches()
gc.collect()
print(f"After jax.clear_caches() + gc.collect: {jax.devices()[0].memory_stats()['bytes_in_use'] / 1e9:.3f} GB")

After allocation: 0.067 GB
After del + gc.collect (no clear_caches): 0.067 GB
After jax.clear_caches() + gc.collect: 0.067 GB


In [ ]:
# ------------------------------------------------------------
# FIX 1: Add jax.clear_caches() + gc.collect() after prep_afqmc
# ------------------------------------------------------------
# In run_afqmc (line 911-913), change:
#
#   nelec_list[ifrag], norb_list[ifrag] = prep_afqmc(...)
#   run_lnoafqmc(options)
#
# to:
#
#   nelec_list[ifrag], norb_list[ifrag] = prep_afqmc(...)
#   jax.clear_caches()   # flush XLA compilation + buffer caches
#   gc.collect()         # let Python finalize any remaining JAX refs
#   run_lnoafqmc(options)
#
# Also add `import gc` at the top of ulno_afqmc.py.
# This alone typically recovers >80% of the GPU memory used by prep_afqmc.

print("Fix 1: call jax.clear_caches() + gc.collect() between prep_afqmc and run_lnoafqmc")

In [9]:
# ------------------------------------------------------------
# ISSUE 2: Uploading the full cderi_clas to GPU for SVD
# ------------------------------------------------------------
# In prep_afqmc lines 450-452:
#
#   cderi_clas = jnp.array(cderi_clas)              # (naux, npair_clas)  -> GPU
#   cderi_clas = compress_cderi_gpu(cderi_clas, ...)  # SVD on GPU
#   cderi_clas = np.array(cderi_clas)               # bring back to CPU
#
# For a typical production run:
#   naux  ~ 5000-30000
#   npair_clas ~ nclas*(nclas+1)/2, nclas can be 50-200 => npair_clas ~ 1275-20100
#
# The GPU needs to hold THREE copies simultaneously:
#   (a) the input  (naux x npair) float64 = up to 30000 x 20100 x 8 bytes ~ 4.8 GB
#   (b) U from SVD (naux x k)  -- even with full_matrices=False, k = min(naux,npair)
#   (c) Vh from SVD (k x npair)
#
# compress_cderi_cpu already exists (line 236) and does the same thing on CPU.
# The SVD is O(naux * npair^2) which is not accelerated much by GPU for this shape.

# Estimate memory for a realistic case
naux_example = 20000
nclas_example = 100
npair_example = nclas_example * (nclas_example + 1) // 2

input_gb  = naux_example * npair_example * 8 / 1e9
# SVD temporaries: U shape (naux, npair), Vh shape (npair, npair)
svd_u_gb  = naux_example * npair_example * 8 / 1e9   # dominant term
svd_vh_gb = npair_example * npair_example * 8 / 1e9

print(f"naux={naux_example}, nclas={nclas_example}, npair={npair_example}")
print(f"  cderi_clas on GPU (input):  {input_gb:.2f} GB")
print(f"  SVD U matrix on GPU:        {svd_u_gb:.2f} GB")
print(f"  SVD Vh matrix on GPU:       {svd_vh_gb:.2f} GB")
print(f"  Total GPU peak during SVD: ~{input_gb + svd_u_gb + svd_vh_gb:.2f} GB")

naux=20000, nclas=100, npair=5050
  cderi_clas on GPU (input):  0.81 GB
  SVD U matrix on GPU:        0.81 GB
  SVD Vh matrix on GPU:       0.20 GB
  Total GPU peak during SVD: ~1.82 GB


In [ ]:
# ------------------------------------------------------------
# FIX 2: Use compress_cderi_cpu instead of compress_cderi_gpu
# ------------------------------------------------------------
# Replace lines 450-452 in prep_afqmc:
#
#   OLD:
#     cderi_clas = jnp.array(cderi_clas)
#     cderi_clas = compress_cderi_gpu(cderi_clas, thresh=chol_cut)
#     cderi_clas = np.array(cderi_clas)
#
#   NEW:
#     cderi_clas = compress_cderi_cpu(cderi_clas, thresh=chol_cut)
#
# This avoids placing the (naux x npair_clas) matrix on the GPU at all.
# numpy's SVD (backed by LAPACK) is perfectly efficient for this operation
# because the matrix is "tall-and-skinny": many rows (naux) but few columns (npair_clas).

# Benchmark CPU vs GPU SVD for a tall-and-skinny matrix
import time

naux_bench  = 5000
npair_bench = 5050   # nclas=100

rng = np.random.default_rng(0)
M_cpu = rng.standard_normal((naux_bench, npair_bench)).astype(np.float64)
M_gpu = jnp.array(M_cpu)

# CPU SVD
t0 = time.perf_counter()
_, s, Vh = np.linalg.svd(M_cpu, full_matrices=False)
t1 = time.perf_counter()
print(f"CPU SVD  ({M_cpu.shape}): {t1-t0:.3f} s")

# GPU SVD
_ = jnp.linalg.svd(M_gpu, full_matrices=False)  # warm-up
t0 = time.perf_counter()
_, s, Vh = jnp.linalg.svd(M_gpu, full_matrices=False)
Vh.block_until_ready()
t1 = time.perf_counter()
print(f"GPU SVD  ({M_gpu.shape}): {t1-t0:.3f} s")

# GPU is rarely faster than CPU for tall-and-skinny SVD,
# but it does occupy GPU memory throughout.

In [ ]:
# ------------------------------------------------------------
# ISSUE 3: @jax.jit compilation cache holds GPU memory
# ------------------------------------------------------------
# The following functions in ulno_afqmc.py are decorated with @jax.jit:
#   jk_from_cderi   (line 92)  - used only in h1e_uas -> get_veff_gpu
#   cderi2mo_gpu    (line 221) - used only in the DF loop in prep_afqmc
#   get_eri         (line 230) - appears unused in production path
#   _svd_gpu        (line 261) - used only in compress_cderi_gpu
#
# Each @jax.jit compilation stores:
#   - the compiled XLA executable (many MB per unique input shape)
#   - static input buffers used during tracing
#
# After prep_afqmc these compiled executables are no longer needed
# until the next fragment's prep_afqmc. They consume GPU memory for
# nothing during run_lnoafqmc.
#
# jax.clear_caches() (Fix 1) also clears these compiled executables,
# so Fix 1 handles this too. But to be precise:

# Show that jit cache grows with each unique shape
@jax.jit
def dummy_matmul(a, b):
    return a @ b

for n in [64, 128, 256]:
    a = jnp.ones((n, n))
    b = jnp.ones((n, n))
    dummy_matmul(a, b).block_until_ready()

mem_before = jax.devices()[0].memory_stats()['bytes_in_use']
jax.clear_caches()
gc.collect()
mem_after = jax.devices()[0].memory_stats()['bytes_in_use']
print(f"GPU memory freed by jax.clear_caches(): {(mem_before - mem_after) / 1e6:.1f} MB")

In [ ]:
# ------------------------------------------------------------
# ISSUE 4: get_veff_gpu accumulates intermediate arrays in the loop
# ------------------------------------------------------------
# In get_veff_gpu (line 113-131):
#
#   vj   = jnp.zeros_like(dm_a)       <- GPU
#   vk_a = jnp.zeros_like(dm_a)       <- GPU
#   vk_b = jnp.zeros_like(dm_b)       <- GPU
#   for cderi in mf.with_df.loop():
#       cderi = jnp.array(lib.unpack_tril(cderi))  <- GPU
#       dvj, dvk_a, dvk_b = jk_from_cderi(cderi, dm_a, dm_b)
#       vj   += dvj     # creates new GPU buffer each time
#       vk_a += dvk_a   # creates new GPU buffer each time
#       vk_b += dvk_b   # creates new GPU buffer each time
#
# JAX uses functional semantics: `vj += dvj` is `vj = vj + dvj`.
# Each iteration creates 3 new (nao, nao) buffers for vj, vk_a, vk_b.
# The old buffers are dead references but their GPU memory is not
# immediately returned to the allocator; it waits for JAX's GC sweep.
#
# For nao=1000 and 50 DF batches:
#   Each iteration leaks 3 x 1000^2 x 8 bytes = 24 MB transiently
#   (recovered eventually, but peak usage is inflated)
#
# This is a minor issue compared to Issues 1-3, but worth noting.

print("Issue 4 is minor for typical nao; resolved indirectly by Fix 1.")

In [ ]:
# ------------------------------------------------------------
# ISSUE 5: In _prep_afqmc, integrals loaded as jnp live on GPU throughout AFQMC
# ------------------------------------------------------------
# _prep_afqmc (line 594-603) reads the FCIDUMP_chol file and creates:
#
#   h1_a    = jnp.array(...).reshape(nmo_a, nmo_a)
#   h1_b    = jnp.array(...).reshape(nmo_b, nmo_b)
#   h1mod_a = jnp.array(...).reshape(nmo_a, nmo_a)
#   h1mod_b = jnp.array(...).reshape(nmo_b, nmo_b)
#   chol_a  = jnp.array(...).reshape(-1, nmo_a, nmo_a)   <- LARGEST
#   chol_b  = jnp.array(...).reshape(-1, nmo_b, nmo_b)   <- LARGEST
#
# These are then assembled into ham_data and wave_data dicts and passed
# to the sampler. This is intentional -- AFQMC needs them on GPU.
#
# BUT: there is a double-copy issue at lines 618-621:
#
#   ham_data["h1"]     = [jnp.array(h1_a), jnp.array(h1_b)]       <- new copies
#   ham_data["h1_mod"] = [jnp.array(h1mod_a), jnp.array(h1mod_b)] <- new copies
#   ham_data["chol"]   = [chol_a.reshape(...), chol_b.reshape(...)] <- reshape
#
# At line 618, `jnp.array(h1_a)` where h1_a is already a jnp.array
# -- this is a NO-OP in JAX (returns the same buffer), so it is fine.
# BUT chol_a at line 602 is (nchol, nmo_a, nmo_a) and at 620 it is
# reshaped to (nchol, nmo_a*nmo_a). jnp reshape is zero-copy for
# contiguous arrays, so this is also fine.
#
# The real issue is that the original h1_a, h1_b, h1mod_a, h1mod_b,
# chol_a, chol_b local variables are NOT deleted after being put into
# ham_data -- they stay alive until _prep_afqmc returns, holding
# redundant GPU references. Add explicit deletes after packing.

print("Issue 5: explicit del of h1_a, h1_b, h1mod_a, h1mod_b, chol_a, chol_b")
print("         after they are packed into ham_data in _prep_afqmc.")

In [ ]:
# ------------------------------------------------------------
# SUMMARY OF ALL FIXES
# ------------------------------------------------------------
#
# Fix 1 (HIGH IMPACT) -- ulno_afqmc.py :: run_afqmc, after line 912:
#   Add `import gc` at top of file.
#   After `prep_afqmc(...)` returns, before `run_lnoafqmc(options)`, add:
#     jax.clear_caches()
#     gc.collect()
#
# Fix 2 (HIGH IMPACT) -- ulno_afqmc.py :: prep_afqmc, lines 450-452:
#   Replace:
#     cderi_clas = jnp.array(cderi_clas)
#     cderi_clas = compress_cderi_gpu(cderi_clas, thresh=chol_cut)
#     cderi_clas = np.array(cderi_clas)
#   With:
#     cderi_clas = compress_cderi_cpu(cderi_clas, thresh=chol_cut)
#   (compress_cderi_cpu is already defined at line 236)
#
# Fix 3 (LOW IMPACT) -- ulno_afqmc.py :: _prep_afqmc, after lines 618-621:
#   Add:
#     del h1_a, h1_b, h1mod_a, h1mod_b, chol_a, chol_b
#     gc.collect()
#
# Together Fix 1 + Fix 2 eliminate the peak GPU usage from prep_afqmc
# so that run_lnoafqmc starts with a nearly empty GPU.

fixes = {
    "Fix 1": "jax.clear_caches() + gc.collect() after prep_afqmc in run_afqmc",
    "Fix 2": "Use compress_cderi_cpu instead of compress_cderi_gpu in prep_afqmc",
    "Fix 3": "del local jnp variables after packing into ham_data in _prep_afqmc",
}
for k, v in fixes.items():
    print(f"{k}: {v}")

In [ ]:
# ------------------------------------------------------------
# Optional: helper to monitor GPU memory during prep_afqmc
# ------------------------------------------------------------
# Insert calls to gpu_mem_gb() at key points in prep_afqmc to
# confirm where the peak occurs.

def gpu_mem_gb(label=""):
    """Print current GPU memory usage via JAX backend."""
    try:
        stats = jax.devices()[0].memory_stats()
        used  = stats.get('bytes_in_use', 0) / 1e9
        limit = stats.get('bytes_limit', 0)  / 1e9
        print(f"[GPU mem] {label}: {used:.3f} / {limit:.3f} GB")
    except Exception as e:
        print(f"[GPU mem] could not query: {e}")

# Usage example (insert into prep_afqmc):
gpu_mem_gb("baseline")

dummy = jnp.ones((1024, 1024), dtype=jnp.float64)
dummy.block_until_ready()
gpu_mem_gb("after 8 MB allocation")

del dummy
jax.clear_caches()
gc.collect()
gpu_mem_gb("after clear_caches + gc.collect")